# Research Tracks Walkthrough (New Steps)

This notebook demonstrates the **new implementation flow**:

1. Run `scripts/05_run_research_tracks.py` (Student-t sweep, fixed-sigma mentor track, ranking-aware training).
2. Load `research_track_summary.csv`.
3. Compare variants by ICE/NLL/RMSE.
4. Load `best_models.json` and inspect the best configurations.

> Tip: set `RUN_EXPERIMENTS = True` to execute training from this notebook.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
EMBEDDINGS = ROOT / "cache" / "t2837_embeddings_v2.pt"
OUT_DIR = ROOT / "outputs" / "research_tracks"
SUMMARY_CSV = OUT_DIR / "research_track_summary.csv"
BEST_JSON = OUT_DIR / "best_models.json"

RUN_EXPERIMENTS = False  # <- set True to run training from notebook
SEED = 42

print(f"ROOT: {ROOT}")
print(f"EMBEDDINGS: {EMBEDDINGS}")
print(f"OUT_DIR: {OUT_DIR}")


In [ ]:
if RUN_EXPERIMENTS:
    if not EMBEDDINGS.exists():
        raise FileNotFoundError(f"Embeddings file not found: {EMBEDDINGS}")

    cmd = [
        "python", "scripts/05_run_research_tracks.py",
        "--embeddings", str(EMBEDDINGS),
        "--out", str(OUT_DIR),
        "--seed", str(SEED),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped training run. Set RUN_EXPERIMENTS=True to run all tracks.")


In [ ]:
if not SUMMARY_CSV.exists():
    raise FileNotFoundError(
        f"Missing {SUMMARY_CSV}. Run the previous cell with RUN_EXPERIMENTS=True "
        "or run scripts/05_run_research_tracks.py from terminal."
    )

df = pd.read_csv(SUMMARY_CSV)
print(f"Loaded {len(df)} rows from {SUMMARY_CSV}")
df.head()


In [ ]:
metrics = ["track", "variant", "head_name", "rmse", "mae", "nll", "ice", "scaled", "a", "b"]
existing = [c for c in metrics if c in df.columns]
leaderboard = df[existing].sort_values(["ice", "nll"], na_position="last")
leaderboard.head(15)


In [ ]:
plot_df = df.dropna(subset=["nll", "ice"]).copy()

fig, ax = plt.subplots(figsize=(7, 5))
for track, sub in plot_df.groupby("track"):
    ax.scatter(sub["nll"], sub["ice"], label=track, s=55, alpha=0.85)

ax.set_xlabel("NLL (lower is better)")
ax.set_ylabel("ICE (lower is better)")
ax.set_title("Research tracks: NLL vs ICE")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
if not BEST_JSON.exists():
    raise FileNotFoundError(f"Missing {BEST_JSON}")

best = json.loads(BEST_JSON.read_text())
print("Best by ICE:")
print(best["best_ice"])
print("\nBest by NLL:")
print(best["best_nll"])


## Next actions

- Freeze the top 2 configurations from this notebook and rerun with multiple seeds.
- Export the selected rows as a report table for your mentor meeting.
- If needed, add the Track D feature-ablation branch into `scripts/05_run_research_tracks.py` and re-run.
